# Advanced Problems with Solutions: Relevant Python 3.8 Changes

This notebook contains advanced, testable practice problems based on key Python 3.8-era features:

- positional-only parameters using `/`
- f-string debugging expressions with `=`
- `as_integer_ratio()` polymorphism
- `functools.lru_cache` used with and without parentheses
- `math.dist`
- `collections.namedtuple(defaults=...)`
- reversed dictionary views
- `continue` in `finally`
- `SyntaxWarning` for `is` with literals

Each problem includes a complete solution and assertions.

## Problem 1 — Design a stable API with positional-only parameters

Create a function `normalize_pair(x, y, /, *, scale=1, as_tuple=True)`.

Requirements:

- `x` and `y` must be positional-only.
- `scale` and `as_tuple` must be keyword-only.
- Return `(x / scale, y / scale)` when `as_tuple=True`.
- Return `{"x": x / scale, "y": y / scale}` when `as_tuple=False`.
- Raise `ValueError` when `scale == 0`.
- Demonstrate that passing `x=` or `y=` as keywords fails.

In [1]:
def normalize_pair(x, y, /, *, scale=1, as_tuple=True):
    if scale == 0:
        raise ValueError("scale cannot be zero")

    normalized_x = x / scale
    normalized_y = y / scale

    if as_tuple:
        return normalized_x, normalized_y

    return {"x": normalized_x, "y": normalized_y}


assert normalize_pair(10, 20) == (10, 20)
assert normalize_pair(10, 20, scale=10) == (1, 2)
assert normalize_pair(10, 20, scale=10, as_tuple=False) == {"x": 1, "y": 2}

try:
    normalize_pair(x=10, y=20)
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError for positional-only arguments")

try:
    normalize_pair(10, 20, scale=0)
except ValueError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected ValueError for zero scale")

normalize_pair(15, 30, scale=15)

Expected error: normalize_pair() got some positional-only arguments passed as keyword arguments: 'x, y'
Expected error: scale cannot be zero


(1.0, 2.0)

### Solution explanation

The slash marks all parameters before it as positional-only:

```python
def normalize_pair(x, y, /, *, scale=1, as_tuple=True):
```

The `*` then makes `scale` and `as_tuple` keyword-only. This is useful when `x` and `y` are conceptually values, not stable public keyword names.

## Problem 2 — Preserve backward compatibility while renaming internals

Suppose you maintain an old public function `distance_2d(a, b)` where callers pass points positionally. You want to rename internal parameters without breaking users.

Implement:

```python
def distance_2d(point_a, point_b, /):
    ...
```

Requirements:

- Accept two 2D points.
- Use `math.dist`.
- Do not allow callers to pass `point_a=` or `point_b=`.
- Return the Euclidean distance.

In [2]:
import math

def distance_2d(point_a, point_b, /):
    if len(point_a) != 2 or len(point_b) != 2:
        raise ValueError("Both points must be 2D")
    return math.dist(point_a, point_b)


assert distance_2d((0, 0), (3, 4)) == 5.0
assert round(distance_2d((1, 1), (4, 5)), 5) == 5.0

try:
    distance_2d(point_a=(0, 0), point_b=(3, 4))
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError for keyword use")

distance_2d((0, 0), (1, 1))

Expected error: distance_2d() got some positional-only arguments passed as keyword arguments: 'point_a, point_b'


1.4142135623730951

### Solution explanation

Positional-only parameters let you change internal parameter names later without breaking keyword-based callers, because no keyword-based callers are allowed in the first place.

## Problem 3 — Build a debug formatter using f-string `=`

Write `debug_summary(name, values)` that returns a multiline string showing:

- the expression text and value for `name`
- the expression text and value for `len(values)`
- the expression text and value for `sum(values)`
- the expression text and value for the average formatted to two decimal places

Use Python 3.8 f-string debugging syntax wherever appropriate.

In [3]:
def debug_summary(name, values):
    average = sum(values) / len(values) if values else 0
    return (
        f"{name=}\n"
        f"{len(values)=}\n"
        f"{sum(values)=}\n"
        f"{average=:.2f}"
    )


result = debug_summary("batch-A", [10, 20, 30])
print(result)

assert "name='batch-A'" in result
assert "len(values)=3" in result
assert "sum(values)=60" in result
assert "average=20.00" in result

name='batch-A'
len(values)=3
sum(values)=60
average=20.00


### Solution explanation

The syntax:

```python
f"{variable=}"
```

prints both the expression text and its representation. Format specifiers still work:

```python
f"{average=:.2f}"
```

## Problem 4 — Convert heterogeneous numeric values to exact fractions

Write `ratios(values)`.

Requirements:

- Accept values that support `.as_integer_ratio()`.
- Return a list of `(numerator, denominator)` tuples.
- Support `bool`, `int`, `float`, `Decimal`, and `Fraction`.
- Raise `TypeError` with a helpful message for unsupported values.

In [4]:
from decimal import Decimal
from fractions import Fraction

def ratios(values):
    result = []

    for value in values:
        try:
            ratio = value.as_integer_ratio()
        except AttributeError as exc:
            raise TypeError(f"{value!r} does not support as_integer_ratio()") from exc

        result.append(ratio)

    return result


values = [True, 12, 0.5, Decimal("0.25"), Fraction(2, 3)]
answer = ratios(values)

assert answer[0] == (1, 1)
assert answer[1] == (12, 1)
assert answer[2] == (1, 2)
assert answer[3] == (1, 4)
assert answer[4] == (2, 3)

try:
    ratios(["0.5"])
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError")

answer

Expected error: '0.5' does not support as_integer_ratio()


[(1, 1), (12, 1), (1, 2), (1, 4), (2, 3)]

### Solution explanation

Python 3.8 made `.as_integer_ratio()` more uniformly available across numeric types such as `bool`, `int`, and `Fraction`, in addition to types like `float` and `Decimal`.

This is a good duck-typing example: ask the object to do the operation instead of checking for every possible numeric class.

## Problem 5 — Use `lru_cache` without parentheses and inspect cache behavior

Create a cached recursive Fibonacci function using `@lru_cache` without parentheses.

Requirements:

- Use the Python 3.8-supported decorator form `@lru_cache`.
- Implement `fib(n)` for `n >= 1`.
- Raise `ValueError` for `n < 1`.
- Prove caching is working with `.cache_info()`.

In [5]:
from functools import lru_cache

@lru_cache
def fib(n):
    if n < 1:
        raise ValueError("n must be >= 1")

    if n <= 2:
        return 1

    return fib(n - 1) + fib(n - 2)


fib.cache_clear()
assert fib(1) == 1
assert fib(2) == 1
assert fib(10) == 55

info = fib.cache_info()
print(info)

assert info.hits > 0
assert info.misses > 0

try:
    fib(0)
except ValueError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected ValueError")

fib(20)

CacheInfo(hits=9, misses=10, maxsize=128, currsize=10)
Expected error: n must be >= 1


6765

### Solution explanation

In Python 3.8, `lru_cache` can be used directly as:

```python
@lru_cache
def fib(...):
```

instead of requiring:

```python
@lru_cache()
def fib(...):
```

The cache still provides introspection helpers such as `.cache_info()` and `.cache_clear()`.

## Problem 6 — Compare manual distance code with `math.dist`

Write `nearest(origin, points)`.

Requirements:

- Use `math.dist`.
- Return the point nearest to `origin`.
- If two points have the same distance, keep the first one.
- Raise `ValueError` when `points` is empty.
- Work for 2D and 3D points.

In [6]:
import math

def nearest(origin, points):
    if not points:
        raise ValueError("points cannot be empty")

    return min(points, key=lambda point: math.dist(origin, point))


assert nearest((0, 0), [(10, 10), (3, 4), (1, 1)]) == (1, 1)
assert nearest((0, 0, 0), [(2, 0, 0), (1, 1, 1), (5, 5, 5)]) == (1, 1, 1)
assert nearest((0, 0), [(1, 0), (0, 1)]) == (1, 0)

try:
    nearest((0, 0), [])
except ValueError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected ValueError")

nearest((0, 0), [(5, 5), (2, 2), (9, 9)])

Expected error: points cannot be empty


(2, 2)

### Solution explanation

`math.dist(p, q)` avoids hand-written square-root formulas and generalizes cleanly across dimensions.

It also avoids easy indexing mistakes in manual formulas.

## Problem 7 — Named tuple defaults and right-to-left application

Create a `Job` named tuple with fields:

```python
id, owner, priority, retries, active
```

Requirements:

- `id` and `owner` are required.
- `priority` defaults to `"normal"`.
- `retries` defaults to `0`.
- `active` defaults to `True`.
- Verify `_field_defaults`.
- Create one full instance and one minimal instance.

In [7]:
from collections import namedtuple

Job = namedtuple(
    "Job",
    "id owner priority retries active",
    defaults=("normal", 0, True),
)

minimal = Job(1001, "alice")
full = Job(1002, "bob", "high", 3, False)

assert minimal == Job(1001, "alice", "normal", 0, True)
assert full == Job(1002, "bob", "high", 3, False)

assert Job._field_defaults == {
    "priority": "normal",
    "retries": 0,
    "active": True,
}

minimal, full, Job._field_defaults

(Job(id=1001, owner='alice', priority='normal', retries=0, active=True),
 Job(id=1002, owner='bob', priority='high', retries=3, active=False),
 {'priority': 'normal', 'retries': 0, 'active': True})

### Solution explanation

Named tuple defaults are applied from right to left. Since there are three defaults, they apply to:

```python
priority, retries, active
```

The required fields are therefore:

```python
id, owner
```

## Problem 8 — Detect shared mutable defaults in named tuples

The following named tuple uses a mutable default incorrectly:

```python
Bad = namedtuple("Bad", "a b c", defaults=([],) * 3)
```

Write code that proves all three fields share the same list object. Then define a safer factory function `make_good()` that returns independent lists.

In [8]:
from collections import namedtuple

Bad = namedtuple("Bad", "a b c", defaults=([],) * 3)

bad = Bad()
bad.a.append("leak")

assert bad.a == ["leak"]
assert bad.b == ["leak"]
assert bad.c == ["leak"]
assert bad.a is bad.b is bad.c

print("Bad shared defaults:", bad)

Good = namedtuple("Good", "a b c")

def make_good():
    return Good([], [], [])

good = make_good()
good.a.append("safe")

assert good.a == ["safe"]
assert good.b == []
assert good.c == []
assert good.a is not good.b
assert good.b is not good.c

good

Bad shared defaults: Bad(a=['leak'], b=['leak'], c=['leak'])


Good(a=['safe'], b=[], c=[])

### Solution explanation

The expression:

```python
([],) * 3
```

does not create three independent lists. It creates a tuple containing the same list reference three times.

For mutable defaults, prefer a factory function that creates fresh objects.

## Problem 9 — Reverse dictionary views without copying first

Write `last_n_items(mapping, n)`.

Requirements:

- Use `reversed(mapping.items())`.
- Return the last `n` key-value pairs as a list.
- Preserve reverse insertion order.
- Return all items if `n` is larger than the mapping size.
- Raise `ValueError` for negative `n`.

In [9]:
def last_n_items(mapping, n):
    if n < 0:
        raise ValueError("n cannot be negative")

    result = []

    for index, item in enumerate(reversed(mapping.items())):
        if index >= n:
            break
        result.append(item)

    return result


data = {"a": 1, "b": 2, "c": 3, "d": 4}

assert last_n_items(data, 0) == []
assert last_n_items(data, 2) == [("d", 4), ("c", 3)]
assert last_n_items(data, 10) == [("d", 4), ("c", 3), ("b", 2), ("a", 1)]

try:
    last_n_items(data, -1)
except ValueError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected ValueError")

last_n_items(data, 3)

Expected error: n cannot be negative


[('d', 4), ('c', 3), ('b', 2)]

### Solution explanation

Python 3.8 allows reversing dictionary views directly:

```python
reversed(mapping.items())
reversed(mapping.keys())
reversed(mapping.values())
```

This avoids manually materializing `list(mapping.items())` just to reverse it.

## Problem 10 — `continue` inside `finally`

Write `collect_valid_numbers(items)`.

Requirements:

- Try to convert each item to `int`.
- Append valid integers.
- Skip invalid values.
- Always increment a `processed` counter in a `finally` block.
- Use `continue` inside the `finally` block when conversion failed.
- Return `(numbers, processed)`.

In [10]:
def collect_valid_numbers(items):
    numbers = []
    processed = 0

    for item in items:
        failed = False

        try:
            number = int(item)
        except (TypeError, ValueError):
            failed = True
        finally:
            processed += 1
            if failed:
                continue

        numbers.append(number)

    return numbers, processed


numbers, processed = collect_valid_numbers(["10", "x", 20, None, "30"])

assert numbers == [10, 20, 30]
assert processed == 5

numbers, processed

([10, 20, 30], 5)

### Solution explanation

Python 3.8 supports `continue` in a `finally` clause. This is valid, but should be used carefully.

In many real codebases, a simpler `except: continue` may be clearer. This problem exists to demonstrate the feature, not to recommend overusing it.

## Problem 11 — Programmatically detect suspicious `is` comparisons with literals

Python 3.8 emits a warning for code such as:

```python
x is 1
```

Write `compile_and_capture_warnings(source)`.

Requirements:

- Compile source code.
- Capture `SyntaxWarning`.
- Return the warning messages as strings.
- Demonstrate that `x is 1` warns but `x == 1` does not.

In [11]:
import warnings

def compile_and_capture_warnings(source):
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", SyntaxWarning)
        compile(source, "<dynamic>", "exec")

    return [
        str(warning.message)
        for warning in captured
        if issubclass(warning.category, SyntaxWarning)
    ]


bad_warnings = compile_and_capture_warnings("x = 1\nx is 1")
good_warnings = compile_and_capture_warnings("x = 1\nx == 1")

print("bad_warnings:", bad_warnings)
print("good_warnings:", good_warnings)

assert any('"is" with a literal' in message for message in bad_warnings)
assert good_warnings == []

bad_warnings: ['"is" with \'int\' literal. Did you mean "=="?']
good_warnings: []


AssertionError: 

### Solution explanation

`is` checks object identity. It should not be used for value comparisons with numbers or strings.

Use:

```python
x == 1
```

not:

```python
x is 1
```

Identity checks are appropriate for singleton objects such as `None`:

```python
x is None
```

## Problem 12 — Integrated mini-project: lightweight metrics report

Build `metrics_report(name, points, /, *, origin=(0, 0), precision=2)`.

Requirements:

- `name` and `points` are positional-only.
- `origin` and `precision` are keyword-only.
- Use `math.dist`.
- Use `namedtuple` with defaults for the report structure.
- Use f-string debugging syntax in a human-readable `debug` string.
- Return a named tuple with fields:

```python
name, count, nearest, farthest, debug
```

Defaults should make `count=0`, `nearest=None`, `farthest=None`, and `debug=""`.

For empty `points`, return the default-style empty report with the provided name.

In [12]:
import math
from collections import namedtuple

MetricsReport = namedtuple(
    "MetricsReport",
    "name count nearest farthest debug",
    defaults=(0, None, None, ""),
)

def metrics_report(name, points, /, *, origin=(0, 0), precision=2):
    points = list(points)

    if not points:
        return MetricsReport(name=name)

    nearest_point = min(points, key=lambda point: math.dist(origin, point))
    farthest_point = max(points, key=lambda point: math.dist(origin, point))

    nearest_distance = math.dist(origin, nearest_point)
    farthest_distance = math.dist(origin, farthest_point)

    debug = (
        f"{name=}\n"
        f"{len(points)=}\n"
        f"{origin=}\n"
        f"{nearest_distance=:.{precision}f}\n"
        f"{farthest_distance=:.{precision}f}"
    )

    return MetricsReport(
        name=name,
        count=len(points),
        nearest=nearest_point,
        farthest=farthest_point,
        debug=debug,
    )


report = metrics_report("sample", [(3, 4), (1, 1), (10, 0)], origin=(0, 0), precision=1)
empty_report = metrics_report("empty", [])

assert report.name == "sample"
assert report.count == 3
assert report.nearest == (1, 1)
assert report.farthest == (10, 0)
assert "nearest_distance=1.4" in report.debug
assert "farthest_distance=10.0" in report.debug

assert empty_report == MetricsReport("empty", 0, None, None, "")

try:
    metrics_report(name="sample", points=[(0, 0)])
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError for positional-only arguments")

report

Expected error: metrics_report() got some positional-only arguments passed as keyword arguments: 'name, points'


MetricsReport(name='sample', count=3, nearest=(1, 1), farthest=(10, 0), debug="name='sample'\nlen(points)=3\norigin=(0, 0)\nnearest_distance=1.4\nfarthest_distance=10.0")

### Solution explanation

This problem combines several Python 3.8-related ideas:

- `/` makes `name` and `points` positional-only.
- `*` makes `origin` and `precision` keyword-only.
- `math.dist` handles distance calculation.
- `namedtuple(..., defaults=...)` provides compact structured results.
- f-string debugging syntax makes diagnostic output concise.

The result is compact, but still readable.

## Final best-practices checklist

Use these features deliberately:

1. Use positional-only parameters when argument names should not become part of the public API.
2. Use f-string debugging syntax for diagnostics, logging experiments, and teaching examples.
3. Use `.as_integer_ratio()` for duck-typed exact numeric conversion.
4. Use `@lru_cache` directly when you do not need custom cache arguments.
5. Prefer `math.dist` over manual Euclidean distance formulas.
6. Be careful with mutable defaults in `namedtuple`.
7. Use reversed dictionary views when you need reverse insertion order.
8. Avoid `is` with literals; use `==` for value comparison.